# 02 - Data Preprocessing
Goal: clean the data, encode text to numbers, split into train/test, and scale.

In [10]:
import pandas as pd
import numpy as np 
import joblib 

# scikit-learn tools for splitting and scaling 
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [11]:
# Load the original file (we never change the raw file itself)
df = pd.read_csv("../data/raw/Telco_Customer_Churn.csv")

print("Shape:", df.shape)

Shape: (7043, 21)


In [24]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

df["TotalCharges"] = df["TotalCharges"].fillna(0)

print("Missing in TotalCharges:", df["TotalCharges"].isnull().sum())
print("data type:", df["TotalCharges"].dtype)

Missing in TotalCharges: 0
data type: float64


In [12]:
# customerID is just a label. It has no link to churn, and could confuse the model.
df = df.drop(columns=["customerID"])

print("Shape now:", df.shape)

Shape now: (7043, 20)


In [13]:
#Yes -> 1 (churned), No -> 0 (stayed)
df["Churn"] = df["Churn"].map({"Yes":1, "No":0})

print(df["Churn"].value_counts())

Churn
0    5174
1    1869
Name: count, dtype: int64


In [18]:
# These columns have 3 values: Yes, No, "No internet service" / "No phone service".
# "No internet service" really just means "No", so we merge them.
# (The fact that the customer has no internet is already stored in InternetService.)
service_cols = ["MultipleLines", "OnlineSecurity", "OnlineBackup",
                "DeviceProtection", "TechSupport", "StreamingTV", "StreamingMovies"]

for col in service_cols:
    df[col] = df[col].replace({"No internet service": "No", "No phone service": "No"})

# Check one column: it should now have only Yes / No
print(df["TechSupport"].unique())

<ArrowStringArray>
['No', 'Yes']
Length: 2, dtype: str


In [19]:
# All these columns now contain only Yes / No
binary_cols = ["Partner", "Dependents", "PhoneService", "PaperlessBilling"] + service_cols

for col in binary_cols:
    df[col] = df[col].map({"Yes": 1, "No": 0})

# gender has Male / Female. We create a single 0/1 column.
df["gender"] = df["gender"].map({"Male": 1, "Female": 0})

df[binary_cols + ["gender"]].head()

,Partner,Dependents,PhoneService,PaperlessBilling,MultipleLines,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,gender
0,1,0,0,1,0,0,1,0,0,0,0,0
1,0,0,1,0,0,1,0,1,0,0,0,1
2,0,0,1,1,0,1,1,0,0,0,0,1
3,0,0,0,0,0,1,0,1,1,0,0,1
4,0,0,1,1,0,0,0,0,0,0,0,0


In [20]:
# These three columns have 3 or more categories
multi_cols = ["InternetService", "Contract", "PaymentMethod"]

# get_dummies creates one 0/1 column per category.
# drop_first=True removes one column per feature (it is not needed, and avoids repetition).
# dtype=int gives 0/1 instead of True/False.
df = pd.get_dummies(df, columns=multi_cols, drop_first=True, dtype=int)

print("Shape now:", df.shape)
print(list(df.columns))

Shape now: (7043, 24)
['gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure', 'PhoneService', 'MultipleLines', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'PaperlessBilling', 'MonthlyCharges', 'TotalCharges', 'Churn', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check', 'PaymentMethod_Mailed check']


In [21]:
# Every column should be int64 or float64. If you see "object", something was missed.
print(df.dtypes.value_counts())

print("\nAny missing values left?", df.isnull().sum().sum())

int64      22
float64     1
str         1
Name: count, dtype: int64

Any missing values left? 0


In [26]:
# X = the inputs (features), y = the answer we want to predict
X = df.drop(columns=["Churn"])
y = df["Churn"]

# Split 80% train / 20% test.
# stratify=y  -> keeps the same churn % in both parts (important for imbalanced data)
# random_state=42 -> gives the same split every time (reproducible)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

print("Train:", X_train.shape, " Test:", X_test.shape)
print("Churn rate in train:", round(y_train.mean() * 100, 1), "%")
print("Churn rate in test: ", round(y_test.mean() * 100, 1), "%")

Train: (5634, 23)  Test: (1409, 23)
Churn rate in train: 26.5 %
Churn rate in test:  26.5 %


In [27]:
# Only these 3 columns have large values. The rest are already 0/1.
num_cols = ["tenure", "MonthlyCharges", "TotalCharges"]

# Work on copies, so the original X_train / X_test stay unchanged
X_train_scaled = X_train.copy()
X_test_scaled = X_test.copy()

scaler = StandardScaler()

# fit_transform on TRAIN: learns the mean/std from train, then scales train
X_train_scaled[num_cols] = scaler.fit_transform(X_train[num_cols])

# transform on TEST: uses the mean/std learned from TRAIN (no fit here!)
X_test_scaled[num_cols] = scaler.transform(X_test[num_cols])

# Check: train columns should have mean ~0 and std ~1
print(X_train_scaled[num_cols].describe().round(2))

        tenure  MonthlyCharges  TotalCharges
count  5634.00         5634.00       5634.00
mean     -0.00           -0.00          0.00
std       1.00            1.00          1.00
min      -1.32           -1.54         -1.01
25%      -0.96           -0.97         -0.83
50%      -0.14            0.18         -0.40
75%       0.92            0.83          0.67
max       1.61            1.79          2.80


In [28]:
# Put the target column back next to the features, so each file is self-contained
train = X_train_scaled.copy()
train["Churn"] = y_train

test = X_test_scaled.copy()
test["Churn"] = y_test

# Save to the processed folder
train.to_csv("../data/processed/train.csv", index=False)
test.to_csv("../data/processed/test.csv", index=False)

# Save the fitted scaler (we need the same one later in the app)
joblib.dump(scaler, "../models/scaler.pkl")

print("Saved train.csv, test.csv, and scaler.pkl")

Saved train.csv, test.csv, and scaler.pkl


## My Notes

1. Fixed TotalCharges by converting it to numeric and filling blanks with 0 because those 11 customers have tenure = 0 and have not been billed yet.
2. Binary columns were mapped to 0/1. One-hot encoding was used for multi-category categorical features: InternetService, Contract, and PaymentMethod.
3. I split before scaling because scaling the whole dataset before splitting causes data leakage from the test set into the training set.
4. Train has 5,634 rows and test has 1,409 rows.